In [1]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 01_Bronze_Ingestion
# Layer: Bronze
#
# Description:
# Load CSV files from the Lakehouse Files folder
# into Bronze Delta tables.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# ETL Audit
# ------------------------------------------------------------

audit_log = []

# ------------------------------------------------------------
# Lakehouse Files Path
# ------------------------------------------------------------

base_path = "Files"

# ------------------------------------------------------------
# Source Files
# ------------------------------------------------------------

files = [
    "dim_customer.csv",
    "dim_product.csv",
    "dim_date.csv",
    "dim_route.csv",
    "dim_carrier.csv",
    "dim_warehouse.csv",
    "fact_shipment.csv",
    "fact_fuel_price.csv",
    "fact_road_event.csv"
]

print("=" * 60)
print("Starting Bronze Ingestion...")
print("=" * 60)

# ------------------------------------------------------------
# Load Files
# ------------------------------------------------------------

for file in files:

    table_name = file.replace(".csv", "")

    print(f"\nLoading {table_name}...")

    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(f"{base_path}/{file}")
    )

    rows_before = df.count()
    rows_after = rows_before
    duplicates_removed = 0

    print(f"Rows: {rows_after}")
    print(f"Columns: {len(df.columns)}")

    (
        df.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable(table_name)
    )

    audit_log.append((
        "01_Bronze_Ingestion",
        "Bronze",
        table_name,
        rows_before,
        rows_after,
        duplicates_removed,
        "Success"
    ))

    print(f"✓ Bronze table created: {table_name}")
    print("-" * 60)

# ------------------------------------------------------------
# Write ETL Audit Log
# ------------------------------------------------------------

audit_schema = StructType([
    StructField("Notebook", StringType(), False),
    StructField("Layer", StringType(), False),
    StructField("TableName", StringType(), False),
    StructField("RowsBefore", LongType(), False),
    StructField("RowsAfter", LongType(), False),
    StructField("DuplicatesRemoved", LongType(), False),
    StructField("Status", StringType(), False)
])

audit_df = spark.createDataFrame(audit_log, audit_schema)

audit_df = audit_df.withColumn(
    "RunTimestamp",
    current_timestamp()
)

(
    audit_df.write
        .mode("append")
        .format("delta")
        .saveAsTable("etl_audit_log")
)

print("\nETL audit successfully written.")

print("\n" + "=" * 60)
print("Bronze Layer successfully created.")
print("=" * 60)

StatementMeta(, 57faea9e-9096-464b-b9d8-8473f84f622c, 3, Finished, Available, Finished, False)

Starting Bronze Ingestion...

Loading dim_customer...
Rows: 800
Columns: 6
✓ Bronze table created: dim_customer
------------------------------------------------------------

Loading dim_product...
Rows: 7
Columns: 7
✓ Bronze table created: dim_product
------------------------------------------------------------

Loading dim_date...
Rows: 731
Columns: 15
✓ Bronze table created: dim_date
------------------------------------------------------------

Loading dim_route...
Rows: 25
Columns: 6
✓ Bronze table created: dim_route
------------------------------------------------------------

Loading dim_carrier...
Rows: 7
Columns: 5
✓ Bronze table created: dim_carrier
------------------------------------------------------------

Loading dim_warehouse...
Rows: 10
Columns: 7
✓ Bronze table created: dim_warehouse
------------------------------------------------------------

Loading fact_shipment...
Rows: 50000
Columns: 17
✓ Bronze table created: fact_shipment
----------------------------------------